# Daily market and macro dataset

Build a 15-year daily dataset for the clustering / Entropy Pooling research project. This notebook downloads and aligns observations and calculates eight trailing S&P 500 technical features; clustering comes later. No upstream Fortitudo code is copied here.

Run all cells from the repository folder. In VS Code, select the repository's `.venv` Python kernel. If dependencies are missing, uncomment the installation cell below.

The cutoff is fixed for reproducibility: **2011-10-05 through 2026-10-04**, using available market observations and excluding the cutoff day's potentially incomplete prices. Change `AS_OF` to refresh the research window.

In [13]:
# Uncomment once if needed, then restart the kernel.
# %pip install pandas numpy yfinance requests ipykernel

## Variables and sources

| Column | Definition | Units | Source |
| --- | --- | --- | --- |
| `sp500` | S&P 500 price index, daily close (not total return) | Index points | [Yahoo Finance: ^GSPC](https://finance.yahoo.com/quote/%5EGSPC/history/) |
| `sp500_gold_ratio` | S&P 500 index divided by gold futures price | Index points per USD/troy ounce | Calculated here: `sp500 / gold` |
| `breakeven_10y` | 10-year breakeven inflation rate | Percent | [FRED: T10YIE](https://fred.stlouisfed.org/series/T10YIE) |
| `real_yield_10y` | 10-year inflation-indexed Treasury constant-maturity yield | Percent | [FRED: DFII10](https://fred.stlouisfed.org/series/DFII10) |
| `slope_10y_2y` | 10-year minus 2-year Treasury yield | Percentage points | [FRED: T10Y2Y](https://fred.stlouisfed.org/series/T10Y2Y) |
| `baa_10y_spread` | Moody’s seasoned Baa corporate bond yield minus 10-year Treasury yield | Percentage points | [FRED: BAA10Y](https://fred.stlouisfed.org/series/BAA10Y) |
| `vix` | Cboe VIX, daily close (30-day horizon) | Annualized volatility percent | [Cboe history](https://www.cboe.com/tradable_products/vix/vix_historical_data/) |
| `vix3m` | Cboe VIX3M, daily close (3-month horizon) | Annualized volatility percent | [Cboe VIX3M](https://www.cboe.com/us/indices/dashboard/vix3m/) |
| `vix_term_spread` | **VIX minus VIX3M** | Volatility percentage points | Calculated here |

Rates and volatility are stored in their published units: `2.5` means 2.5%, not 0.025. These are levels, yields and spreads, alongside eight dimensionless technical indicators. FRED's S&P 500 series has only 10 years of history, hence the separate equity source. No API key is needed.
Gold uses Yahoo’s `GC=F` futures history, not a spot fixing or a gold ETF. The ratio uses same-date closes; futures roll changes and different closing times can affect it. Gold closes are retained in the raw download for calculating the ratio; only `sp500_gold_ratio` enters the exported feature datasets. Missing gold values produce missing ratios, and no filling is applied.

The export contains **17 features**: the nine market/macro series above plus the eight technical indicators defined below. Raw OHLC and gold inputs remain in the source downloads. Technical indicators are calculated from S&P 500 OHLC history, before aligning macro dates or selecting complete cases.


In [14]:
from pathlib import Path
from io import StringIO
import json
import warnings

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

AS_OF = pd.Timestamp("2026-10-05")  # exclusive endpoint; edit to refresh
START = AS_OF - pd.DateOffset(years=15)
END = AS_OF - pd.Timedelta(days=1)
# Buffer equity history so the 252-observation lookback is ready at START.
EQUITY_DOWNLOAD_START = START - pd.DateOffset(years=2)

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
yf.set_tz_cache_location(str(DATA_DIR / "yfinance_cache"))
retrieved_at = pd.Timestamp.now(tz="UTC").isoformat()
print(f"Requested window: {START.date()} to {END.date()}")

Requested window: 2011-10-05 to 2026-10-04


## 1. Download S&P 500 OHLC and gold closes

Use the S&P 500 price index itself and Yahoo’s `GC=F` gold futures history as a gold-price proxy. Save both source tables locally. Equity is downloaded with two extra years of history for technical-indicator warmup; only the requested 15-year window is exported. A failed download stops the notebook rather than silently changing the asset.


In [15]:
equity = yf.download(
    "^GSPC",
    start=EQUITY_DOWNLOAD_START.strftime("%Y-%m-%d"),
    end=AS_OF.strftime("%Y-%m-%d"),  # Yahoo's end is exclusive
    interval="1d",
    auto_adjust=False,
    multi_level_index=False,
    progress=False,
    threads=False,
    timeout=30,
)
if equity is None or equity.empty:
    raise RuntimeError("S&P 500 download failed. Check the connection and rerun this cell.")
equity.index = pd.DatetimeIndex(equity.index).tz_localize(None).normalize()
equity.to_csv(RAW_DIR / "yahoo_sp500.csv", index_label="date")
sp500 = pd.to_numeric(equity["Close"], errors="coerce").dropna().rename("sp500")
gold_prices = yf.download(
    "GC=F",
    start=START.strftime("%Y-%m-%d"),
    end=AS_OF.strftime("%Y-%m-%d"),
    interval="1d",
    auto_adjust=False,
    multi_level_index=False,
    progress=False,
    threads=False,
    timeout=30,
)
if gold_prices is None or gold_prices.empty:
    raise RuntimeError("Gold download failed. Check the connection and rerun this cell.")
gold_prices.index = pd.DatetimeIndex(gold_prices.index).tz_localize(None).normalize()
gold_prices.to_csv(RAW_DIR / "yahoo_gold.csv", index_label="date")
gold = pd.to_numeric(gold_prices["Close"], errors="coerce").rename("gold")
if (gold.dropna() <= 0).any():
    raise ValueError("Observed gold prices must be positive to calculate the ratio.")


## 2. Calculate eight S&P 500 technical features

All windows count **observed equity trading rows**, including today's close. Let `P` be the unadjusted S&P 500 close and `r = log(P / P.shift(1))`. Calculate on the buffered OHLC history **before** trimming to the research window. Do not fill missing prices or warmup values.

| Column | Definition | Units / convention |
| --- | --- | --- |
| `sp500_momentum_63d` | `log(P / P.shift(63))` | Decimal log return; approximately 3 months |
| `sp500_drawdown_252d` | `P / rolling_max(P, 252) - 1` | Decimal drawdown from the trailing high; 0 at the high |
| `sp500_realized_vol_21d` | `std(r, 21, ddof=0) * sqrt(252) * 100` | Annualized volatility percent, matching VIX units |
| `sp500_volatility_expansion_21_126d` | `log(std(r, 21) / std(r, 126))` | Dimensionless; positive when recent volatility is higher |
| `sp500_trend_efficiency_21d` | `abs(log(P / P.shift(21))) / rolling_sum(abs(r), 21)` | Log-return path efficiency from 0 to 1; a flat path is 0 |
| `sp500_rsi_14d` | `100 * Wilder_avg(gains, 14) / (Wilder_avg(gains, 14) + Wilder_avg(losses, 14))` | RSI from 0 to 100; gains/losses use price changes |
| `sp500_ma_gap_200d` | `P / SMA(P, 200) - 1` | Decimal relative distance from the 200-day average |
| `sp500_atrp_14d` | `100 * Wilder_ATR(14) / P` | Average true range percent of the current close |

Wilder averages start with the simple average of the first 14 valid consecutive observations, then update as `(13 * previous_average + current_value) / 14`. RSI needs 14 price changes (15 closes); a flat series has RSI 50, gains only 100, and losses only 0. True range is `max(High - Low, abs(High - previous_close), abs(Low - previous_close))`; only the first downloaded row uses `High - Low` when no previous close exists. Later missing inputs restart the Wilder warmup. Undefined volatility ratios stay missing.

These are trailing descriptors available **after the current close**. Shift them before use in any strategy that acts earlier. They are kept in their documented units here and will enter `02` through its existing `StandardScaler`, alongside all other selected features. Definitions follow [StockCharts RSI](https://chartschool.stockcharts.com/table-of-contents/technical-indicators-and-overlays/technical-indicators/relative-strength-index-rsi) and [Fidelity's Wilder ATR reference](https://www.fidelity.com/products/atp/content/wsFuncRef_US.pdf); the log-return efficiency definition is stated explicitly above.


In [16]:
TECHNICAL_FEATURE_SPECS = {
    "sp500_momentum_63d": {"definition": "log(close / close.shift(63))", "units": "decimal log return", "lookback": 63},
    "sp500_drawdown_252d": {"definition": "close / close.rolling(252).max() - 1", "units": "decimal drawdown", "lookback": 252},
    "sp500_realized_vol_21d": {"definition": "std(daily log returns, 21, ddof=0) * sqrt(252) * 100", "units": "annualized volatility percent", "lookback": 21},
    "sp500_volatility_expansion_21_126d": {"definition": "log(std(log returns, 21) / std(log returns, 126)); ddof=0", "units": "dimensionless log ratio", "lookback": [21, 126]},
    "sp500_trend_efficiency_21d": {"definition": "abs(log(close / close.shift(21))) / rolling_sum(abs(daily log returns), 21)", "units": "dimensionless, 0 to 1", "lookback": 21},
    "sp500_rsi_14d": {"definition": "100 * Wilder_avg(positive price changes, 14) / Wilder_avg(absolute price changes, 14)", "units": "RSI points, 0 to 100", "lookback": 14, "smoothing": "Wilder; simple-average seed"},
    "sp500_ma_gap_200d": {"definition": "close / close.rolling(200).mean() - 1", "units": "decimal relative gap", "lookback": 200},
    "sp500_atrp_14d": {"definition": "100 * Wilder_avg(true range, 14) / close", "units": "percent of close", "lookback": 14, "smoothing": "Wilder; simple-average seed"},
}


def wilder_average(values, period=14):
    """Simple-average seed, then Wilder recursion; missing input resets warmup."""
    values = pd.to_numeric(values, errors="coerce").astype(float)
    values = values.where(np.isfinite(values))
    seeds = values.rolling(period, min_periods=period).mean().to_numpy()
    result = np.full(len(values), np.nan)
    previous = np.nan
    for i, value in enumerate(values.to_numpy()):
        if not np.isfinite(value):
            previous = np.nan
        elif np.isnan(previous):
            previous = seeds[i]
        else:
            previous = ((period - 1) * previous + value) / period
        result[i] = previous
    return pd.Series(result, index=values.index, name=values.name)


def calculate_sp500_technicals(ohlc):
    """Causal, scale-invariant trailing indicators on the full equity calendar."""
    if not ohlc.index.is_unique or not ohlc.index.is_monotonic_increasing:
        raise ValueError("Equity dates must be unique and sorted before calculating indicators.")
    prices = ohlc[["Close", "High", "Low"]].apply(pd.to_numeric, errors="coerce")
    if not (np.isfinite(prices) | prices.isna()).to_numpy().all():
        raise ValueError("Observed OHLC values must be finite.")
    close, high, low = prices["Close"], prices["High"], prices["Low"]
    if (close.dropna() <= 0).any() or (high < low).any():
        raise ValueError("S&P 500 closes must be positive and High must be at least Low.")
    returns = np.log(close / close.shift(1))
    vol21 = returns.rolling(21, min_periods=21).std(ddof=0)
    vol126 = returns.rolling(126, min_periods=126).std(ddof=0)
    path_length = returns.abs().rolling(21, min_periods=21).sum()
    efficiency = (np.log(close / close.shift(21)).abs() / path_length).mask(path_length.eq(0), 0)

    changes = close.diff()
    average_gain = wilder_average(changes.clip(lower=0), 14)
    average_loss = wilder_average((-changes).clip(lower=0), 14)
    average_total = average_gain + average_loss
    rsi = (100 * average_gain / average_total).mask(average_total.eq(0), 50)

    previous_close = close.shift(1)
    true_range = pd.concat([
        high - low, (high - previous_close).abs(), (low - previous_close).abs()
    ], axis=1).max(axis=1, skipna=False)
    if len(true_range):
        true_range.iloc[0] = high.iloc[0] - low.iloc[0]
    atr = wilder_average(true_range, 14)

    return pd.DataFrame({
        "sp500_momentum_63d": np.log(close / close.shift(63)),
        "sp500_drawdown_252d": close / close.rolling(252, min_periods=252).max() - 1,
        "sp500_realized_vol_21d": vol21 * np.sqrt(252) * 100,
        "sp500_volatility_expansion_21_126d": np.log((vol21 / vol126).where((vol21 > 0) & (vol126 > 0))),
        "sp500_trend_efficiency_21d": efficiency.clip(0, 1),
        "sp500_rsi_14d": rsi,
        "sp500_ma_gap_200d": close / close.rolling(200, min_periods=200).mean() - 1,
        "sp500_atrp_14d": 100 * atr / close,
    }, index=ohlc.index)


pre_window_equity_rows = int(equity.loc[equity.index < START, "Close"].notna().sum())
if pre_window_equity_rows < 252:
    raise ValueError("Download at least 252 earlier equity observations for indicator warmup.")
technicals = calculate_sp500_technicals(equity)
print(f"Equity warmup observations before START: {pre_window_equity_rows:,}")
display(technicals.loc[START:END].head())


Equity warmup observations before START: 505


,sp500_momentum_63d,sp500_drawdown_252d,sp500_realized_vol_21d,sp500_volatility_expansion_21_126d,sp500_trend_efficiency_21d,sp500_rsi_14d,sp500_ma_gap_200d,sp500_atrp_14d
Date,,,,,,,,
2011-10-05,-0.167930,-0.161028,30.526915,0.161725,0.051786,45.885661,-0.105030,2.877053
2011-10-06,-0.142806,-0.145672,29.545574,0.123565,0.082629,49.861520,-0.088329,2.811149
2011-10-07,-0.132747,-0.152646,29.463775,0.120178,0.076001,48.131934,-0.095405,2.762521
2011-10-10,-0.094748,-0.123730,30.204664,0.127905,0.099304,55.088089,-0.064309,2.716376
2011-10-11,-0.097305,-0.123254,30.147511,0.126685,0.082466,55.194769,-0.063573,2.592314


## 3. Download Treasury variables, credit spread and volatility indices

FRED uses missing values on some holidays. Keep them as `NaN`. Download VIX and VIX3M directly from Cboe's CSV endpoints.

In [17]:
def download_csv(url, filename, params=None):
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    table = pd.read_csv(StringIO(response.text), na_values=["."])
    (RAW_DIR / filename).write_text(response.text, encoding="utf-8")
    return table

fred_ids = {
    "breakeven_10y": "T10YIE",
    "real_yield_10y": "DFII10",
    "slope_10y_2y": "T10Y2Y",
    "baa_10y_spread": "BAA10Y",
}
series = {"sp500": sp500, "gold": gold}
source_urls = {
    "sp500": "https://finance.yahoo.com/quote/%5EGSPC/history/",
    "gold": "https://finance.yahoo.com/quote/GC%3DF/history/",
}

for name, series_id in fred_ids.items():
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv"
    table = download_csv(url, f"fred_{series_id}.csv", params={
        "id": series_id,
        "cosd": START.strftime("%Y-%m-%d"),
        "coed": END.strftime("%Y-%m-%d"),
    })
    dates = pd.to_datetime(table.iloc[:, 0])
    values = pd.to_numeric(table[series_id], errors="coerce")
    series[name] = pd.Series(values.to_numpy(), index=dates, name=name)
    source_urls[name] = f"https://fred.stlouisfed.org/series/{series_id}"

for name, index_id in {"vix": "VIX", "vix3m": "VIX3M"}.items():
    url = f"https://cdn.cboe.com/api/global/us_indices/daily_prices/{index_id}_History.csv"
    table = download_csv(url, f"cboe_{index_id}.csv")
    table.columns = table.columns.str.strip().str.upper()
    dates = pd.to_datetime(table["DATE"], format="%m/%d/%Y")
    values = pd.to_numeric(table["CLOSE"], errors="coerce")
    series[name] = pd.Series(values.to_numpy(), index=dates, name=name)
    source_urls[name] = url
# Each indicator joins the same alignment and data-quality pipeline.
series.update({column: technicals[column] for column in TECHNICAL_FEATURE_SPECS})


## 4. Align on S&P 500 trading dates

`daily_levels` retains all S&P 500 trading dates in the requested research window, including missing macro observations. No forward-fill, backward-fill or interpolation is applied. `daily_complete` keeps only dates with all 17 features observed. This avoids inventing price or yield changes, but excludes some equity trading days when the Treasury market is closed.

If calculating daily equity returns later, calculate them on `daily_levels` **before** filtering complete rows; otherwise a return could span more than one trading day.

In [18]:
for name, values in series.items():
    values = values.sort_index().loc[START:END]
    if values.index.has_duplicates:
        raise ValueError(f"Duplicate dates in {name}")
    observed = values.dropna()
    if observed.empty or observed.index.min() > START + pd.Timedelta(days=7):
        raise ValueError(f"{name} does not cover the beginning of the 15-year window")
    if observed.index.max() < END - pd.Timedelta(days=7):
        warnings.warn(f"{name} ends at {observed.index.max().date()}; check source freshness")
    series[name] = values

all_dates = pd.concat(series.values(), axis=1).sort_index()
daily_levels = all_dates.reindex(series["sp500"].dropna().index).copy()
daily_levels.index.name = "date"
daily_levels["vix_term_spread"] = daily_levels["vix"] - daily_levels["vix3m"]
daily_levels["sp500_gold_ratio"] = daily_levels["sp500"] / daily_levels["gold"]
# Gold is a source input for the ratio, not a separate clustering feature.
daily_levels = daily_levels.drop(columns=["gold"])
base_columns = [
    "sp500", "breakeven_10y", "real_yield_10y", "slope_10y_2y", "baa_10y_spread",
    "vix", "vix3m", "vix_term_spread", "sp500_gold_ratio",
]
daily_levels = daily_levels[base_columns + list(TECHNICAL_FEATURE_SPECS)]
daily_complete = daily_levels.dropna().copy()

assert not daily_complete.empty
assert daily_levels.shape[1] == 17
assert daily_complete[list(TECHNICAL_FEATURE_SPECS)].nunique().gt(1).all()
assert daily_complete["sp500_drawdown_252d"].between(-1, 0).all()
assert daily_complete["sp500_trend_efficiency_21d"].between(0, 1).all()
assert daily_complete["sp500_rsi_14d"].between(0, 100).all()
assert daily_complete[["sp500_realized_vol_21d", "sp500_atrp_14d"]].ge(0).all().all()
assert daily_levels.index.is_unique and daily_levels.index.is_monotonic_increasing
assert np.isfinite(daily_complete.to_numpy()).all()
assert (daily_complete[["sp500", "sp500_gold_ratio", "vix", "vix3m"]] > 0).all().all()
assert np.allclose(
    daily_complete["vix_term_spread"],
    daily_complete["vix"] - daily_complete["vix3m"],
)

assert np.allclose(
    daily_complete["sp500_gold_ratio"],
    daily_complete["sp500"] / series["gold"].reindex(daily_complete.index),
)

quality = pd.DataFrame({
    "observed_rows": daily_levels.count(),
    "missing_rows": daily_levels.isna().sum(),
    "first_observed": daily_levels.apply(lambda s: s.first_valid_index()),
    "last_observed": daily_levels.apply(lambda s: s.last_valid_index()),
})
print(f"Equity trading dates: {len(daily_levels):,}")
print(f"Complete dates: {len(daily_complete):,}")
print(f"Dates excluded from complete dataset: {len(daily_levels) - len(daily_complete):,}")
display(quality)
display(daily_complete.head())
display(daily_complete.tail())

Equity trading dates: 3,770
Complete dates: 3,739
Dates excluded from complete dataset: 31


/var/folders/gd/46g2hsl52yx0m_gmp4131fdh0000gp/T/ipykernel_10642/2688671855.py:12: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  all_dates = pd.concat(series.values(), axis=1).sort_index()


,observed_rows,missing_rows,first_observed,last_observed
sp500,3770,0,2011-10-05,2026-10-02
breakeven_10y,3742,28,2011-10-05,2026-10-02
real_yield_10y,3742,28,2011-10-05,2026-10-02
slope_10y_2y,3742,28,2011-10-05,2026-10-02
baa_10y_spread,3740,30,2011-10-05,2026-10-02
vix,3770,0,2011-10-05,2026-10-02
vix3m,3770,0,2011-10-05,2026-10-02
vix_term_spread,3770,0,2011-10-05,2026-10-02
sp500_gold_ratio,3767,3,2011-10-05,2026-10-02
sp500_momentum_63d,3770,0,2011-10-05,2026-10-02


,sp500,breakeven_10y,real_yield_10y,slope_10y_2y,baa_10y_spread,vix,vix3m,vix_term_spread,sp500_gold_ratio,sp500_momentum_63d,sp500_drawdown_252d,sp500_realized_vol_21d,sp500_volatility_expansion_21_126d,sp500_trend_efficiency_21d,sp500_rsi_14d,sp500_ma_gap_200d,sp500_atrp_14d
date,,,,,,,,,,,,,,,,,
2011-10-05,1144.030029,1.80,0.12,1.67,3.36,37.81,38.31,-0.50,0.697452,-0.167930,-0.161028,30.526915,0.161725,0.051786,45.885661,-0.105030,2.877053
2011-10-06,1164.969971,1.88,0.13,1.72,3.34,36.27,37.04,-0.77,0.705230,-0.142806,-0.145672,29.545574,0.123565,0.082629,49.861520,-0.088329,2.811149
2011-10-07,1155.459961,1.94,0.16,1.80,3.32,36.20,37.31,-1.11,0.706920,-0.132747,-0.152646,29.463775,0.120178,0.076001,48.131934,-0.095405,2.762521
2011-10-11,1195.540039,1.95,0.23,1.86,3.32,32.86,34.06,-1.20,0.720335,-0.097305,-0.123254,30.147511,0.126685,0.082466,55.194769,-0.063573,2.592314
2011-10-12,1207.250000,1.98,0.26,1.95,3.33,31.26,32.97,-1.71,0.718046,-0.080819,-0.114666,30.168926,0.125858,0.084255,57.168540,-0.054211,2.530000


,sp500,breakeven_10y,real_yield_10y,slope_10y_2y,baa_10y_spread,vix,vix3m,vix_term_spread,sp500_gold_ratio,sp500_momentum_63d,sp500_drawdown_252d,sp500_realized_vol_21d,sp500_volatility_expansion_21_126d,sp500_trend_efficiency_21d,sp500_rsi_14d,sp500_ma_gap_200d,sp500_atrp_14d
date,,,,,,,,,,,,,,,,,
2026-09-28,7683.689941,2.34,2.90,0.32,1.46,16.07,18.23,-2.16,1.843319,0.032171,-0.014784,10.595611,-0.200738,0.051456,51.030376,0.065785,0.871521
2026-09-29,7670.839844,2.35,2.91,0.37,1.46,16.04,18.09,-2.05,1.835261,0.022608,-0.016432,10.578860,-0.201147,0.044916,49.877251,0.063424,0.853506
2026-09-30,7651.540039,2.36,2.93,0.41,1.47,16.34,18.37,-2.03,1.827583,0.022242,-0.018906,10.554167,-0.157722,0.038352,48.118553,0.060197,0.861138
2026-10-01,7666.450195,2.36,2.88,0.46,1.49,16.39,18.58,-2.19,1.824346,0.024188,-0.016995,10.282390,-0.181555,0.040668,49.597177,0.061646,0.861401
2026-10-02,7722.720215,2.36,2.92,0.45,1.47,15.31,18.01,-2.70,1.855397,0.024285,-0.009779,10.460854,-0.166765,0.063327,54.829343,0.068768,0.875640


## 5. Save local datasets

The CSV files, raw downloads and metadata stay in the ignored `data/` directory. The notebook is the reproducible recipe; source data retain their providers' terms. These are currently available historical observations, not an archive of what was known on every past trading date. In a predictive experiment, account for source publication times and use features only when available.

In [19]:
daily_levels.to_csv(DATA_DIR / "daily_macro_levels.csv")
daily_complete.to_csv(DATA_DIR / "daily_macro_complete.csv")
quality.to_csv(DATA_DIR / "data_quality.csv")

metadata = {
    "retrieved_at_utc": retrieved_at,
    "as_of_exclusive": AS_OF.date().isoformat(),
    "requested_start": START.date().isoformat(),
    "requested_end": END.date().isoformat(),
    "equity_download_start": EQUITY_DOWNLOAD_START.date().isoformat(),
    "pre_window_equity_rows": pre_window_equity_rows,
    "feature_columns": list(daily_levels.columns),
    "technical_features": TECHNICAL_FEATURE_SPECS,
    "technical_feature_policy": {
        "calendar": "equity trading observations; calculate before macro filtering",
        "availability": "after current close; shift for earlier decisions",
        "warmup": "two-year equity buffer; full rolling windows; no filling",
        "wilder_seed": "simple average of first 14 consecutive valid observations",
        "missing_inputs": "preserve missing values; reset Wilder warmup",
        "annualization_trading_days": 252,
        "realized_vol_ddof": 0,
        "references": [
            "https://chartschool.stockcharts.com/table-of-contents/technical-indicators-and-overlays/technical-indicators/relative-strength-index-rsi",
            "https://www.fidelity.com/products/atp/content/wsFuncRef_US.pdf",
        ],
    },
    "calendar": "observed S&P 500 trading dates",
    "missing_value_policy": "no filling; separate complete-case dataset",
    "term_spread": "VIX - VIX3M",
    "sp500_gold_ratio": "sp500 / gold; same-date closes; no filling",
    "gold_instrument": "Yahoo GC=F gold futures close (USD per troy ounce), price proxy",
    "rows_levels": len(daily_levels),
    "rows_complete": len(daily_complete),
    "first_complete_date": daily_complete.index.min().date().isoformat(),
    "last_complete_date": daily_complete.index.max().date().isoformat(),
    "sources": source_urls,
    "versions": {"pandas": pd.__version__, "numpy": np.__version__, "yfinance": yf.__version__},
}
(DATA_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(f"Saved datasets and metadata to {DATA_DIR.resolve()}")

Saved datasets and metadata to /Users/lajosgalambos/Desktop/Quantitative_Investment_Management 2/Chapters/clustering-entropy-pooling/data


## Monthly earnings: assessment for a later extension

**A useful candidate for a slow-moving corporate state, provided point-in-time data are available.** Carry forward the latest published earnings value onto each daily date, starting only when it became available to the strategy. Use the release timestamp (or next trading day for an after-close release), not the month the figure describes. Keep the original release date and the age of the observation. Do not backward-fill or interpolate between releases.

A monthly observation repeated across daily rows supplies context, not new daily information. Fifteen years provide roughly 180 monthly updates, so persistent values can influence cluster assignments for many consecutive days. Evaluate their incremental value against the daily-only baseline with chronological validation and account for serial dependence. Revised realized earnings and historical analyst estimates require their original vintages; a latest-history download does not establish what investors knew at the time. [FRED/ALFRED explains the vintage distinction](https://fred.stlouisfed.org/docs/api/fred/realtime_period.html).

For the first clustering experiment, keep the daily-only dataset above. Later, choose a specific earnings measure and source before adding it.

## Notes for the next research step

- Keep these market levels and calculated indicators as the source dataset. Consider equity returns or trailing returns rather than the drifting S&P 500 index level for regime clustering; choose transformations to match the research question.
- VIX, VIX3M and their difference are exactly linearly dependent. Retain all three in the dataset, but avoid using all three as equally weighted clustering inputs without justification: that would repeatedly weight the volatility dimension.
- Scale chosen features using training-period statistics only. Evaluate clusters chronologically before integrating them into the Fortitudo scenario / EP pipeline.
- Momentum, RSI, efficiency, drawdown and the moving-average gap overlap in their information; adding dimensions also changes the weight of the equity-technical family in K-means. Compare cluster profiles and stability against the previous nine-feature baseline.
